# 分词
传统NlP分词，是把文本按空格，或者标点符号等分隔符进行切分，得到一个个单词或词组的过程。对于中文来说，分词是一个重要的预处理步骤，因为中文文本没有明显的单词边界。

但在大模型中，分词是将自然语言与Token对应起来的过程，大模型使用的分词器通常是基于BPE（Byte Pair Encoding）或者WordPiece算法，这些算法能够有效地处理未知词汇和稀有词汇。基于字节的压缩算法（BPE）通过将常见的字符组合替换为单个Token来减少词汇表的大小，而WordPiece算法则通过将单词分解成更小的子词来处理未知词汇。

## 传统分词
jieba使用方式：

    1.精确模式：试图将句子最精确地切开，适合文本分析。

    2.全模式：把句子中所有的可能的词语都扫描

    3.搜索引擎模式：在精确模式的基础上，对长词再次切分，提高召回率，适合搜索引擎分词。

还可以自定义词典，以及词频调整，来提高分词的准确性。

In [4]:
import jieba
from torch.nn import Embedding, parameter

text = "我来到北京清华大学"
# 精确
seg_list = jieba.lcut(text, cut_all=False)
print("精确模式: " + "/ ".join(seg_list))

# 全模式
seg_list = jieba.lcut(text, cut_all=True)
print("全模式: " + "/ ".join(seg_list))

# 搜索引擎模式
seg_list = jieba.cut_for_search(text)
print("搜索引擎模式: " + "/ ".join(seg_list))

# 添加词语
jieba.add_word("自然语言处理")

# 设置词频
jieba.suggest_freq("清华大学",tune=True)


# 自定义词典
# 格式为: 词语 词频
# jieba.load_userdict("userdict.txt")

# 分析一句话的关键词 使用TF-IDF算法
import jieba.analyse

text = "我来到北京清华大学，清华大学是中国最好的大学之一。"
tags = jieba.analyse.extract_tags(text, topK=5,withWeight=True)
print([(tag, weight) for tag, weight in tags])

# 分析一句话的关键词 使用TextRank算法
text_rank = jieba.analyse.textrank(text, topK=5, withWeight=True)
print([(tag, weight) for tag, weight in text_rank])


# 词性标注
import jieba.posseg as jpos
text = "我来到北京清华大学"
seg = jpos.cut(text)
print([(word, pos) for word, pos in seg])


jieba.add_word('你')
jieba.add_word('好')

print(jieba.lcut('你好'))

Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\86191\AppData\Local\Temp\jieba.cache
Loading model cost 0.566 seconds.
Prefix dict has been built successfully.


精确模式: 我/ 来到/ 北京/ 清华大学
全模式: 我/ 来到/ 北京/ 清华/ 清华大学/ 华大/ 大学
搜索引擎模式: 我/ 来到/ 北京/ 清华/ 华大/ 大学/ 清华大学
[('清华大学', 2.020148680405), ('最好', 0.7224368012375), ('大学', 0.71427665281875), ('来到', 0.67321086051375), ('北京', 0.58342528859)]
[('来到', 1.0), ('大学', 1.0), ('北京', 0.994132857417495), ('中国', 0.994132857417495)]
[('我', 'r'), ('来到', 'v'), ('北京', 'ns'), ('清华大学', 'nt')]
['你好']


## Tokenizer？
Tokenizer（分词器）可以将原始文本（raw text）转换为模型能够理解的数字序列，在模型输入和输出的两个主要阶段中发挥重要作用：
### 一、模型输入（编码 Encode）阶段

1. 分词（Tokenize）:将文本拆分为词元（Token），常见的分词方式包括字级、词级、子词级（如 BPE、WordPiece）、空格分词等。
2. 映射（Mapping）:将每个词元映射为词汇表中的唯一 ID，生成的数字序列即为模型的输入。

### 二、模型输出（解码 Decode）阶段

1. 反映射（De-mapping）:将模型输出的数字序列（ID）转换回对应的词元。
2. 文本重组:将解码后的词元以某种规则重新拼接为完整文本。。

In [ ]:
# 将自然语言转为向量的过程
from transformers import AutoTokenizer
import torch.nn as nn
import torch

text = '我爱深度学习，所以我在学习BPE算法'

# 使用gpt2中的BPE算法
tokenizer = AutoTokenizer.from_pretrained('gpt2')

# 转为与token对应的id序列 有三个结果：ids、段id、attention_mask(哪些是真实 Token，哪些是填充 0)填充时会填0
input_ids = tokenizer(text, return_tensors='pt')
# 查看结果（你最关心的 Token ID）
print("Token ID 序列：", input_ids["input_ids"])
# 解码：ID → 文本（验证）
print("还原文本：", tokenizer.decode(input_ids["input_ids"][0]))

# 查看BPE词表大小
print(tokenizer.vocab_size)

class MyEmbedding(nn.Module):
    def __init__(self,vocab_size,dim):
        """
        :param vocab_size: the dictionary size
        :param dim: the dimension of the embedding vector
        """
        super().__init__()
        self.matrix = nn.Parameter(torch.arange(0,vocab_size*dim,dtype=float).reshape([vocab_size,dim]))

    def forward(self, ids):
        return self.matrix[ids]


# num_embedding: 表示词表大小
# embedding_dim: 表示每个词向量的维度
# padding_idx: 用于文本对齐，有些句子长短不一
embed = MyEmbedding(50257,32)

embedding_vector = embed(input_ids['input_ids'])
print(embedding_vector)  # 输出: torch.Size([3, 32])

## 大模型分词
大模型分词通常使用基于BPE或WordPiece算法的分词器，我的理解是压缩自然语言为Token，然后使用id代替。也就是先encode变为id序列，然后再decode变回文本。大模型分词器通常会有一个预训练的词汇表，包含了常见的单词、子词和特殊Token，这些Token可以是单个字符、多个字符的组合，或者是特殊的标记（如[CLS]、[SEP]等）。分词器会根据输入文本中的字符和词汇表中的Token进行匹配，将文本切分成对应的Token序列，并将这些Token转换为对应的id序列，以供模型使用。

### BPE算法
将词转为字节流格式，使用2字节表示字符，然后统计文本中所有的字节对，找到出现频率最高的字节对，将其合并为一个新的Token，重复这个过程直到达到预定的词汇表大小。如果有特殊的词汇，可以将其添加到词汇表中，或者调整词频来提高分词的准确性。

In [13]:
# BPE算法实现
from collections import OrderedDict,defaultdict

class BPETokenizer:
    def __init__(self):
        self.next_id = 0 # 下一个Token的ID
        self.b2i = OrderedDict() # 字节对到ID的映射 映射
        self.i2b = OrderedDict() # ID到字节对的映射 反映射
        self.merges = [] # 保留合并规则


        self.s2i = OrderedDict() # special token
        self.i2s = OrderedDict()

    def get_most_frequent_pair(self, tokens_list):
        # 统计字节对的频率，找到出现频率最高的字节对
        pair_fre = defaultdict(int)
        # 遍历每一条 token 序列
        for seq in tokens_list:
            # 遍历每条序列里相邻的 token 对
            for i in range(len(seq) - 1):
                pair = (seq[i], seq[i+1])
                pair_fre[pair] += 1

        if not pair_fre:
            return None
        return max(pair_fre, key=pair_fre.get)

    def merge_pair(self, pair):
        # 将出现频率最高的字节对合并为一个新的Token
        token_a, token_b = pair
        new_token = token_a + token_b
        self.b2i[new_token] = self.next_id
        self.i2b[self.next_id] = new_token
        self.next_id += 1
        return new_token

    def train(self,texts_list,vocab_size):
        # 单字节id
        for i in range(256):
            self.b2i[bytes([i])] = i
            self.i2b[i] = bytes([i])
        self.next_id = 256

        # 将语料转为字节码
        tokens_list = []
        for text in texts_list:
            token = [bytes([b]) for b in text.encode('utf-8')]
            tokens_list.append(token)

        while True:
            if self.next_id >= vocab_size:
                break

            # 统计字节对的频率，找到出现频率最高的字节对
            most_frequent_pair = self.get_most_frequent_pair(tokens_list)
            if not most_frequent_pair:
                break
            # 将该相邻token组成新token并编号
            new_token = self.merge_pair(most_frequent_pair)
            self.merges.append(most_frequent_pair)   # 记录合并顺序
            token_a,token_b = most_frequent_pair
            print('='*20)
            print(f"合并{token_a},{token_b} ======> {new_token}")
            # 将原token序列中的合并上述产生的新token
            new_tokens_list = []
            for seq in tokens_list:
                new_seq = []
                i = 0
                while i < len(seq):
                    # 如果当前位置是要合并的对 → 替换成新 token
                    if i < len(seq)-1 and seq[i] == token_a and seq[i+1] == token_b:
                        new_seq.append(new_token)
                        i += 2  # 跳过下一个
                    else:
                        new_seq.append(seq[i])
                        i += 1
                new_tokens_list.append(new_seq)

            # 更新 token 列表，进入下一轮合并
            tokens_list = new_tokens_list

    def tokenize(self, text):
        """
        输入一句话 → 输出 BPE 分词后的 token ids
        """
        # 1. 先转成单字节序列
        tokens = [bytes([b]) for b in text.encode("utf-8")]

        for (a, b) in self.merges:
            new_tokens = []
            i = 0
            while i < len(tokens):
                if i < len(tokens) - 1 and tokens[i] == a and tokens[i+1] == b:
                    new_tokens.append(a + b)
                    i += 2
                else:
                    new_tokens.append(tokens[i])
                    i += 1
            tokens = new_tokens

        # 转换为 ID
        token_ids = [self.b2i[t] for t in tokens]
        return token_ids, tokens

    def encode(self, text):
        """
        自然语言 → token ids
        """
        token_ids, _ = self.tokenize(text)
        return token_ids

    def decode(self, token_ids):
        """
        token ids → 自然语言（支持中英文）
        """
        bytes_list = []
        for idx in token_ids:
            bytes_list.append(self.i2b[idx])

        # 把所有 bytes 拼起来 → 解码为字符串
        full_bytes = b''.join(bytes_list)
        return full_bytes.decode('utf-8', errors='replace')

if __name__ == "__main__":
    corpus = [
        'I love deep learning',
        '我爱深度学习',
        '我爱自然语言处理,所以我在学习大模型'
    ]

    tokenizer = BPETokenizer()
    # 设置 min_frequency=2 可以避免在极小语料上合并成整句 token，
    # 这里为了演示保留为 1，你可以自行调整观察结果。
    tokenizer.train(corpus, vocab_size=500)

    # 测试句子
    text = "I love deep learning"
    ids = tokenizer.encode(text)
    print("\n原文：", text)
    print("Token IDs：", ids)
    print("Tokens 序列：", [tokenizer.i2b[i] for i in ids])

    decoded_text = tokenizer.decode(ids)
    print("解码后：", decoded_text)

    # 测试中文
    text2 = "我爱深度学习"
    ids2 = tokenizer.encode(text2)
    print("\n原文：", text2)
    print("Token IDs：", ids2)
    print("Tokens 序列：", [tokenizer.i2b[i] for i in ids2])
    print("解码后：", tokenizer.decode(ids2))

合并b'\xe6',b'\x88' ======> b'\xe6\x88'
合并b'\xe6\x88',b'\x91' ======> b'\xe6\x88\x91'
合并b' ',b'l' ======> b' l'
合并b'\xe6\x88\x91',b'\xe7' ======> b'\xe6\x88\x91\xe7'
合并b'\xe6\x88\x91\xe7',b'\x88' ======> b'\xe6\x88\x91\xe7\x88'
合并b'\xe6\x88\x91\xe7\x88',b'\xb1' ======> b'\xe6\x88\x91\xe7\x88\xb1'
合并b'\xe5',b'\xad' ======> b'\xe5\xad'
合并b'\xe5\xad',b'\xa6' ======> b'\xe5\xad\xa6'
合并b'\xe5\xad\xa6',b'\xe4' ======> b'\xe5\xad\xa6\xe4'
合并b'\xe5\xad\xa6\xe4',b'\xb9' ======> b'\xe5\xad\xa6\xe4\xb9'
合并b'\xe5\xad\xa6\xe4\xb9',b'\xa0' ======> b'\xe5\xad\xa6\xe4\xb9\xa0'
合并b'\xe5',b'\xa4' ======> b'\xe5\xa4'
合并b'I',b' l' ======> b'I l'
合并b'I l',b'o' ======> b'I lo'
合并b'I lo',b'v' ======> b'I lov'
合并b'I lov',b'e' ======> b'I love'
合并b'I love',b' ' ======> b'I love '
合并b'I love ',b'd' ======> b'I love d'
合并b'I love d',b'e' ======> b'I love de'
合并b'I love de',b'e' ======> b'I love dee'
合并b'I love dee',b'p' ======> b'I love deep'
合并b'I love deep',b' l' ======> b'I love deep l'
合并b'I love deep l',b'e' 

In [ ]:
# wordPiece算法实现
from collections import OrderedDict,defaultdict
class WordPieceTokenizer:
    def __init__(self,max_vocab_size):
        # 1. 特殊Token（BERT标准）
        self.special_tokens = ['[PAD]', '[UNK]', '[CLS]', '[SEP]', '[MASK]']

        # 2. 双向映射表
        self.token2id = OrderedDict()  # token → 数字id
        self.id2token = OrderedDict()  # 数字id → token
        self._init_special_tokens()  # 初始化特殊token

        # 3. 合并规则 (a,b) → ab
        self.merges = defaultdict()

        # 4. 频率统计
        self.word_freq = defaultdict()    # 单词频率
        self.char_freq = defaultdict()    # 单字符频率

        # 5. 词表配置
        self.max_vocab_size = max_vocab_size  # 最大词表

        self.next_id = len(self.special_tokens)

    def _init_special_tokens(self):
        # 给特殊token分配id
        for idx, token in enumerate(self.special_tokens):
            self.token2id[token] = idx
            self.id2token[idx] = token

    def _seq_pre_process(self,texts):
        # 语料预处理
        # 按空格分词
        words = []
        for text in texts:
            words.extend(text.strip().split())

        # 统计单词频率 + 拆分字符
        splits = {}
        for word in words:
            self.word_freq[word] += 1
            chars = list(word)
            splits[word] = chars
            # 统计字符频率
            for c in chars:
                self.char_freq[c] += 1
                if c not in self.token2id:
                    self.token2id[c] = self.next_id
                    self.id2token[self.next_id] = c
                    self.next_id += 1
        return splits

    def _compute_score(self,splited):
        """
        :return:
        """
        pair_freq = defaultdict(int)
        # 计算字符与词的得分 score = 配对次数 / (字符a次数 × 字符b次数)
        for word,chars in splited.items():
            if len(chars) < 2:
                continue
            for i in range(len(chars)-1):
                a, b = chars[i], chars[i+1]
                pair_freq[(a, b)] += self.word_freq[word]
        
        max_score = -1
        best_pair = None
        for (a,b),cnt in pair_freq.items():
            cur_score = cnt/(self.char_freq[a]*self.char_freq[b])
            if cur_score > max_score:
                best_pair = (a,b)
                max_score = cur_score
        return best_pair
    
    def _merge_pair(self,best_pair,splited):
        a,b = best_pair
        new_token = a+b

        # 加入token
        if new_token not in self.token2id:
            self.token2id[new_token] = self.next_id
            self.id2token[self.next_id] = new_token
            self.next_id += 1
        self.merges[(a, b)] = new_token

        # 全局替换
        new_splited = defaultdict()
        for word,chars in splited.items():
            new_chars = []
            i = 0
            while i<len(chars):
                if i<len(chars)-1 and chars[i] == a and chars[i+1] == b:
                    new_chars.append(new_token)
                    i+=2
                else:
                    new_chars.append(chars[i])
                    i+=1

            new_splited[word] = new_chars
            # 更新字符频率
            self.char_freq[new_token] += self.word_freq[word]
        return new_splited

    def train(self,texts):
        splited = self._seq_pre_process(texts)
        # 迭代合并直到词表满
        while len(self.token2id) < self.max_vocab_size:
            best_pair = self._compute_score(splited)
            if not best_pair:
                break
            splited = self._merge_pair(best_pair, splited)
        print("训练完成！词表大小:", len(self.token2id))
    
    def tokenizer(self,text):
        tokens = []
        words = text.split()

        for word in words:
            chars = list(word)
            while True:
                merged = False
                for i in range(len(chars)-1):
                    pair = (chars[i], chars[i+1])
                    if pair in self.merges:
                        new_t = self.merges[pair]
                        chars = chars[:i] + [new_t] + chars[i+2:]
                        merged = True
                        break
                if not merged:
                    break
            
            # BERT风格：非词首加 ##
            if len(chars) > 0:
                tokens.append(chars[0])
                for t in chars[1:]:
                    tokens.append(f"##{t}")
        return tokens
    

    def encode(self,text):
        tokens = self.tokenizer(text=text)
        return [self.token2id.get(t, self.token2id['[UNK]']) for t in tokens]

    def decode(self,ids):
        tokens = [self.id2token.get(i, '[UNK]') for i in ids]
        res = ""
        for t in tokens:
            if t.startswith("##"):
                res += t[2:]
            else:
                res += " " + t
        return res.strip()
